# Real-Time Audio Deepfake Inference Engine

Run real-time inference on any audio file (.wav, .flac) to detect authentic human speech vs. synthetic deepfake voice.

In [1]:
from pathlib import Path
import numpy as np
import soundfile as sf
import torch
import torch.nn as nn
import torchaudio
from transformers import Wav2Vec2Model

BASE_DIR = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
MODEL_PATH = BASE_DIR / "models" / "best_wav2vec2_deepfake_model.pth"
device = "cuda" if torch.cuda.is_available() else "cpu"

class AudioDeepfakeDetector:
    def __init__(self, model_path=MODEL_PATH, dev=device):
        self.device = dev
        self.bundle = torch.load(model_path, map_location=self.device, weights_only=False)
        self.target_sr = self.bundle.get("preprocessing_config", {}).get("sample_rate", 16000)
        self.target_samples = self.bundle.get("preprocessing_config", {}).get("target_samples", 64000)
        
        class Classifier(nn.Module):
            def __init__(self):
                super().__init__()
                self.wav2vec2 = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")
                self.classifier = nn.Sequential(
                    nn.LayerNorm(768),
                    nn.Linear(768, 256),
                    nn.GELU(),
                    nn.Dropout(0.3),
                    nn.Linear(256, 2)
                )
            def forward(self, x):
                return torch.softmax(self.classifier(self.wav2vec2(input_values=x).last_hidden_state.mean(dim=1)), dim=-1)

        self.model = Classifier().to(self.device)
        self.model.load_state_dict(self.bundle["model_state_dict"])
        self.model.eval()

    def predict(self, audio_path):
        data, sr = sf.read(str(audio_path), dtype="float32")
        if data.ndim > 1: data = data.mean(axis=1)
        if sr != self.target_sr:
            t_data = torch.from_numpy(data).unsqueeze(0)
            data = torchaudio.transforms.Resample(sr, self.target_sr)(t_data).squeeze(0).numpy()
        if len(data) < self.target_samples: data = np.pad(data, (0, self.target_samples - len(data)))
        else: data = data[:self.target_samples]
        m = np.max(np.abs(data))
        if m > 1e-6: data = (data / m) * 0.95
        
        tensor_in = torch.from_numpy(data).unsqueeze(0).to(self.device)
        with torch.no_grad():
            probs = self.model(tensor_in).cpu().numpy()[0]
        pred_id = int(np.argmax(probs))
        label = "Fake" if pred_id == 1 else "Real"
        
        return {
            "file": Path(audio_path).name,
            "prediction": label,
            "confidence": float(probs[pred_id]),
            "is_fake": bool(pred_id == 1),
            "probabilities": {"Real": float(probs[0]), "Fake": float(probs[1])}
        }

detector = AudioDeepfakeDetector()


e:\Deepfake\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 211/211 [00:00<00:00, 30113.24it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
# Test Inference on Samples
data_dir = BASE_DIR / "data" / "processed_clean"
real_file = list((data_dir / "Real_audio").glob("*.wav"))[0]
fake_file = list((data_dir / "FAKE").glob("*.wav"))[0]

res_real = detector.predict(real_file)
print("="*50)
print(f"Real Sample:     {res_real['file']}")
print(f"Prediction:      {res_real['prediction']} ({res_real['confidence']*100:.2f}% confidence)")
print(f"Is Deepfake:     {res_real['is_fake']}")
print(f"Probabilities:   {res_real['probabilities']}")
print("="*50)

res_fake = detector.predict(fake_file)
print(f"Fake Sample:     {res_fake['file']}")
print(f"Prediction:      {res_fake['prediction']} ({res_fake['confidence']*100:.2f}% confidence)")
print(f"Is Deepfake:     {res_fake['is_fake']}")
print(f"Probabilities:   {res_fake['probabilities']}")
print("="*50)


Real Sample:     PA_T_0000010.wav
Prediction:      Real (100.00% confidence)
Is Deepfake:     False
Probabilities:   {'Real': 0.9999827146530151, 'Fake': 1.731811062199995e-05}
Fake Sample:     LA_T_1000137.wav
Prediction:      Fake (99.99% confidence)
Is Deepfake:     True
Probabilities:   {'Real': 9.036626579472795e-05, 'Fake': 0.9999096393585205}
